# 02 · Preprocessing

Turns raw WESAD pickles into fixed-length, labelled windows.

| Step | Why |
|---|---|
| Keep chest **ECG, EDA, EMG, Temp, Resp** | Respiration is one of the strongest stress markers |
| Anti-aliased decimation 700 Hz → **100 Hz** | v1 used `x[::10]`, which aliases ECG; 100 Hz keeps R-peaks |
| **Per-subject z-score** | Removes person-specific offsets (skin conductance level, body temp) so the model generalises to new people |
| 60 s windows, 15 s hop, inside a single protocol block | Labels are clean; overlap gives ~4× more training windows |

The logic lives in `src/data.py`; this notebook runs it and sanity-checks the output.

In [ ]:
import sys
from pathlib import Path

# Works whether the notebook is run from the repo root or from notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from src.config import WESAD_DIR, PROCESSED_FILE, CHANNELS, CLASS_NAMES, FS
from src.data import build_dataset, load_subject, preprocess_subject, make_windows

print('WESAD dir :', WESAD_DIR, '(exists)' if WESAD_DIR.exists() else '(MISSING - see data/README.md)')

## One subject, step by step

In [ ]:
sig_raw, lab_raw = load_subject('S2')
sig, lab = preprocess_subject(sig_raw, lab_raw)
X_s2, y_s2 = make_windows(sig, lab)
print(f'raw       : {sig_raw.shape} @ 700 Hz')
print(f'processed : {sig.shape} @ {FS} Hz')
print(f'windows   : {X_s2.shape}')
print('per class :', {CLASS_NAMES[k]: v for k, v in sorted(Counter(y_s2).items())})

In [ ]:
fig, axes = plt.subplots(len(CHANNELS), 1, figsize=(14, 2 * len(CHANNELS)), sharex=True)
for cls, color in [(0, 'steelblue'), (1, 'crimson')]:
    w = X_s2[np.flatnonzero(y_s2 == cls)[0]]
    t = np.arange(len(w)) / FS
    for ax, ch, i in zip(axes, CHANNELS, range(len(CHANNELS))):
        ax.plot(t, w[:, i], color=color, lw=0.7, label=CLASS_NAMES[cls])
        ax.set_ylabel(ch)
axes[0].legend(loc='upper right'); axes[-1].set_xlabel('Time (s)')
plt.suptitle('S2 — one normalised 60 s window: Baseline vs Stress', fontweight='bold')
plt.tight_layout(); plt.show()

## Build the full dataset

In [ ]:
build_dataset()

In [ ]:
d = np.load(PROCESSED_FILE)
X, y, subjects = d['X'], d['y'], d['subjects']
print('X', X.shape, X.dtype, f'{X.nbytes/1e6:.0f} MB')
print('class totals:', {CLASS_NAMES[k]: v for k, v in sorted(Counter(y).items())})
print('subjects    :', sorted(set(subjects), key=lambda s: int(s[1:])))